# [skip-ci]

# Building a reference: fetch a panel or build your own

**Utility:** ViralScan detects only viruses present in its reference. You can (1) pull
the curated viral annotation panel from Zenodo, or (2) build a combined host+virus
kallisto index tailored to your study. The combined reference is what makes
multimapping correction possible (host and virus in one index).

`[skip-ci]`: fetching and index building need network and `kb ref`.

## Option 1 — fetch the bundled viral annotation panel

`viralscan data fetch` downloads the viral GTF panel from Zenodo (SHA-256 verified)
into `~/.cache/viralscan/`. This provides the viral gene annotations, not a full
index.

In [ ]:
!viralscan data fetch            # add --cache-dir DIR to relocate; --force to refresh

In [ ]:
from pathlib import Path
cache = Path.home() / '.cache' / 'viralscan'
if cache.exists():
    gtfs = list(cache.rglob('*.gtf'))
    print(f'{len(gtfs)} viral GTFs cached under {cache}')
    for g in gtfs[:5]:
        print(' ', g.name)
else:
    print('Run `viralscan data fetch` first.')

## Option 2 — build a combined host+virus index with `build-ref`

`build-ref` downloads a host cDNA FASTA+GTF from Ensembl and viral sequences from
NCBI, concatenates them, and runs `kb ref`. Give NCBI an email (Entrez requirement).

### A tiny toy build (a couple of small viral genomes, no host)
Fast enough to try; not for real analysis (no host means no host-virus disambiguation).

In [ ]:
!viralscan build-ref \
    --virus-accessions NC_045512.2 NC_004718.3 \  # SARS-CoV-2 + SARS-CoV-1 control
    --output ref_toy/ \
    --ncbi-email you@example.org

### The real, host-aware build (recommended)

Adds the human transcriptome so host-virus ambiguous reads can be resolved by EM.
This is large (human cDNA index) — run it once on a workstation/cluster, then reuse
`ref_human/index.idx` + `ref_human/t2g.txt` for every sample.

In [ ]:
!viralscan build-ref \
    --host human \
    --virus-accessions NC_045512.2 NC_007605.1 NC_001664.4 \  # + EBV, HHV-6A ...
    --output ref_human/ \
    --ncbi-email you@example.org

## Verify the index before running

Check that the viral accessions you expect are present as targets in the t2g file.

In [ ]:
from pathlib import Path
t2g = Path('ref_human/t2g.txt')
if t2g.exists():
    lines = t2g.read_text().splitlines()
    print(f'{len(lines)} targets in index')
    viral = [l for l in lines if 'NC_045512' in l or 'NC_007605' in l]
    print('viral targets found:', len(viral))
else:
    print('Build a reference first.')

## Summary

- `viralscan data fetch` → curated viral GTF panel (annotations).
- `viralscan build-ref --host human --virus-accessions ...` → combined index.
- Reference **annotation coverage** affects which viral genes are counted (the paper's
  EBV LMP-1/EBNA attribution differences trace to annotation, not the aligner).
- Always report the exact reference accessions used.